Heading Measurement Code

Realtime

In [1]:
from pathlib import Path
import csv
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.svm import SVC
import datetime
import sunposition as sp
from pysolar.solar import get_azimuth
from zoneinfo import ZoneInfo
import time
import gc
from concurrent.futures import ThreadPoolExecutor


Dataset Directory Setup

In [ ]:
dataset_dir = Path(
    r"Path\to...\2026-06-17_19-34-06"
)

# Capture Subfolders
capture_folders = sorted([
    f for f in dataset_dir.iterdir()
    if f.is_dir()
])

# Raw File Paths
raw_files = []
for folder in capture_folders:
    files = list(folder.glob("*.raw"))
    if len(files) > 0:
        raw_files.append(files[0])
print("Number of RAW files:", len(raw_files))

# Results CSV Setup
results_csv = dataset_dir / "Results.csv"
with open(results_csv, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([
        'Time',
        'RAW_File',
        'Psi_mes_1',
        'Psi_mes_2'
    ])


Number of RAW files: 20


Yaw Measurement

In [3]:
# Iterate Over Raw Files

# Step 1 Parameters
T = 30
TOP_CROP    = 150
BOTTOM_CROP = 150
LEFT_CROP   = 200
RIGHT_CROP  = 200
    
# SVM Parameters (single-pass, minimal-sample: speed prioritized over accuracy)
N_SAMPLES = 50   # 10/class. SVM fit time is sub-ms regardless of N, so this buys no
                 # extra speed below here - it's kept small only because the fit itself
                 # was never the bottleneck. Too few points does risk a single noisy
                 # pixel dominating the estimate; 20 is a safe practical floor.
C_SVM     = 1.0
rng       = np.random.default_rng()

# Lat and Long
HomeLat = 35.7331
HomeLon = 51.4866

# Image size
width  = 1224 * 2
height = 1024 * 2

# Disable the cyclic GC for the hot loop: per-frame array allocation was triggering
# collections mid-frame, causing large intermittent stalls (measured: std 79ms with a
# 357ms spike vs std 2ms with GC off, on identical repeated work). Collected manually
# and off the critical path instead (see below).
gc.disable()

# Overlap disk I/O with compute: read is I/O-bound and releases the GIL while
# waiting, so reading the NEXT file on a background thread while THIS frame is
# being processed hides most of that latency behind compute we're already paying
# for. Steady-state per-frame cost becomes roughly max(read, compute), not their sum.
def _load_raw(path):
    return np.fromfile(path, dtype=np.uint16).reshape(height, width)

_io_pool = ThreadPoolExecutor(max_workers=1)
next_future = _io_pool.submit(_load_raw, raw_files[0]) if raw_files else None

for i, raw_path in enumerate(raw_files):

    print(f"\nProcessing: {raw_path.name}")
    t_start = time.perf_counter()

    ### Load RAW Image (prefetched on a background thread during the previous
    ### frame's processing - t_read below now reflects a wait, not a full read)
    img = next_future.result()
    t_read = time.perf_counter()

    # start the NEXT file's read now, so it overlaps with this frame's processing
    if i + 1 < len(raw_files):
        next_future = _io_pool.submit(_load_raw, raw_files[i + 1])

    # Polarization Channels (strided views instead of fancy indexing - no gather copy)
    I90_16  = img[0::2, 0::2].astype(np.float32)
    I45_16  = img[0::2, 1::2].astype(np.float32)
    I0_16   = img[1::2, 1::2].astype(np.float32)
    I135_16 = img[1::2, 0::2].astype(np.float32)

    # Stokes Parameters
    S0 = 0.5 * (I0_16 + I45_16 + I90_16 + I135_16)
    S1 = I0_16 - I90_16
    S2 = I45_16 - I135_16

    # DoLP and AoLP
    DoLP = np.sqrt(S1**2 + S2**2) / (S0 + 1e-6)
    AOLP = 0.5 * np.arctan2(S2, S1)
        
    # AoLP Transformation
    AoLP_l = np.array(AOLP)
    iy, ix = np.indices(AoLP_l.shape)
    cy = (AOLP.shape[0] - 1) / 2.0
    cx = (AOLP.shape[1] - 1) / 2.0
    dj = iy - cy
    di = ix - cx
    AoLP_l += np.arctan2(-di, dj)
    AoLP_l = (AoLP_l) % np.pi - np.pi/2

    # Degrees
    AoLP_deg = np.degrees(AoLP_l)
    t_aolp = time.perf_counter()

    ### Step 1: AoLP Threshold Segmentation 

    # Image Size
    h, w = AoLP_deg.shape

    # Rectangular Valid Mask
    valid_mask = np.zeros((h, w), dtype=bool)
    valid_mask[
        TOP_CROP : h - BOTTOM_CROP,
        LEFT_CROP : w - RIGHT_CROP
    ] = True

    # AoLP Filtering
    r1 = (np.abs(AoLP_deg) < T) & valid_mask
    r0 = (np.abs(AoLP_deg) >= T) & valid_mask
    
    ### Step 2: Connected Components
    r0_uint8 = (r0.astype(np.uint8) * 255)
    kernel = np.ones((3, 3), np.uint8)
    r0_eroded = cv2.erode(
        r0_uint8,
        kernel,
        iterations=1
    )
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(r0_eroded)

    # Largest Components (vectorized area lookup - replaces the per-label Python loop,
    # which rescanned the whole image once per label and was the main bottleneck)
    areas = stats[1:, cv2.CC_STAT_AREA]
    order = np.argsort(areas)[::-1]
    r2 = labels == (order[0] + 1)
    r3 = labels == (order[1] + 1)
    
    ### Step 3: Sign Unification
    AoLP_step3 = np.zeros_like(AoLP_l)
    AoLP_step3[r2] = -AoLP_l[r2]
    AoLP_step3[r3] =  AoLP_l[r3]

    # Degrees
    AoLP_step3_deg = np.degrees(AoLP_step3)
    t_cc = time.perf_counter()

    ### SVM Symmetry Axis Estimation

    # Data Processing
    h, w = AoLP_step3.shape
    iy, ix = np.indices((h, w))
    cx = (w - 1) / 2.0
    cy = (h - 1) / 2.0
    labels_map = np.zeros_like(
        AoLP_step3,
        dtype=np.int32
    )    
    labels_map[AoLP_step3 > 0] =  1
    labels_map[AoLP_step3 < 0] = -1
    
    X_all_raw = np.column_stack(
        (ix.ravel(), iy.ravel())
    ).astype(np.float64)
    X_all_raw[:, 0] -= cx
    X_all_raw[:, 1] -= cy
    y_all = labels_map.ravel()
    valid_idx = y_all != 0
    X_valid = X_all_raw[valid_idx]
    y_valid = y_all[valid_idx]
    idx_pos = np.where(y_valid ==  1)[0]
    idx_neg = np.where(y_valid == -1)[0]

    # Single-pass random sampling (no iterative refinement, no averaging)
    n_half = min(N_SAMPLES // 2, len(idx_pos), len(idx_neg))  # guard against tiny frames
    sel_pos = rng.choice(idx_pos, n_half, replace=False)
    sel_neg = rng.choice(idx_neg, n_half, replace=False)
    idx_sel = np.concatenate([sel_pos, sel_neg])
    X_sub = X_valid[idx_sel]
    y_sub = y_valid[idx_sel]

    svm = SVC(kernel='linear', C=C_SVM)
    svm.fit(X_sub, y_sub)

    w_vec = svm.coef_[0]
    theta_rad = np.arctan2(-w_vec[0], w_vec[1])
    theta_final_deg = np.degrees(theta_rad)

    theta_final_rad = np.radians(theta_final_deg)
    t_svm = time.perf_counter()
    print(f"\nFinal angle (single-pass, n={2*n_half} samples): {theta_final_deg:.4f}°")

    # # Final Line Visualization
    # fig_orig, ax_orig = plt.subplots(
    #     figsize=(7, 6)
    # )
    # im2 = ax_orig.imshow(
    #     AoLP_deg,
    #     cmap='jet',
    #     vmin=-90,
    #     vmax=90
    # )
    # plt.colorbar(
    #     im2,
    #     ax=ax_orig,
    #     label='AoLP (deg)'
    # )
    # line_len = (
    #     min(
    #         AoLP_deg.shape[0],
    #         AoLP_deg.shape[1]
    #     ) * 0.75
    # )
    # cx_orig = (
    #     AoLP_deg.shape[1] - 1
    # ) / 2.0
    # cy_orig = (
    #     AoLP_deg.shape[0] - 1
    # ) / 2.0
    # dx = line_len * np.cos(theta_final_rad)
    # dy = line_len * np.sin(theta_final_rad)
    # ax_orig.plot(
    #     [cx_orig - dx, cx_orig + dx],
    #     [cy_orig - dy, cy_orig + dy],
    #     linewidth=2.5,
    #     label=f'Final axis: {theta_final_deg:.2f}°'
    # )
    # ax_orig.set_title(
    #     'Original AoLP with Final Estimated Symmetry Axis'
    # )
    # ax_orig.legend(loc='upper right')
    # ax_orig.axis('off')
    # plt.tight_layout()
    # plt.show()
    
    ### Sun Position + Yaw Estimation

    session_folder = raw_path.parent.parent.name
    capture_folder = raw_path.parent.name
    session_date = session_folder.split('_')[0]
    timestamp_str = f"{session_date} {capture_folder}"

    dt_local = datetime.datetime.strptime(
        timestamp_str,
        "%Y-%m-%d %H-%M-%S.%f"
    )

    dt_local = dt_local.replace(
        tzinfo=ZoneInfo("Asia/Tehran")
    )

    dt_utc = dt_local.astimezone(
        datetime.timezone.utc
    )

    Psi_sn, zen = sp.sunpos(
        dt_utc,
        HomeLat,
        HomeLon,
        0,
        jit=False
    )[:2]

    Psi_sc = (90 - theta_final_deg) % 360
    Psi_sc = Psi_sc % 180
    
    Psi_raw = (Psi_sn - Psi_sc) % 360

    Psi_mes_1 = Psi_raw
    Psi_mes_2 = (Psi_raw + 180) % 360

    print("Yaw candidates (180° ambiguity):")
    print(f"Psi_mes_1 = {Psi_mes_1:.3f}°")
    print(f"Psi_mes_2 = {Psi_mes_2:.3f}°")

    ### Save Results to CSV

    global_time = dt_local.strftime("%H-%M-%S.%f")[:-3]

    with open(results_csv, 'a', newline='') as csvfile:

        writer = csv.writer(csvfile)

        writer.writerow([
            global_time,
            raw_path.name,
            Psi_mes_1,
            Psi_mes_2
        ])

    plt.close('all')

    # Manual GC every 50 frames, placed after this frame's result is already saved -
    # bounds memory growth over a long run without stalling a time-critical frame.
    if (i + 1) % 50 == 0:
        gc.collect()

    t_end = time.perf_counter()
    print(
        f"  [timing]  read={1000*(t_read-t_start):.1f}ms  "
        f"aolp={1000*(t_aolp-t_read):.1f}ms  "
        f"cc+svm={1000*(t_svm-t_aolp):.1f}ms  "
        f"sunpos+io={1000*(t_end-t_svm):.1f}ms  "
        f"TOTAL={1000*(t_end-t_start):.1f}ms"
    )
    print("-" * 50)

_io_pool.shutdown(wait=True)


Processing: polarized_001.raw

Final angle (single-pass, n=50 samples): 79.3211°
Yaw candidates (180° ambiguity):
Psi_mes_1 = 291.062°
Psi_mes_2 = 111.062°
  [timing]  read=162.6ms  aolp=115.3ms  cc+svm=111.9ms  sunpos+io=38.1ms  TOTAL=427.9ms
--------------------------------------------------

Processing: polarized_002.raw

Final angle (single-pass, n=50 samples): 79.8919°
Yaw candidates (180° ambiguity):
Psi_mes_1 = 291.636°
Psi_mes_2 = 111.636°
  [timing]  read=0.5ms  aolp=148.6ms  cc+svm=71.2ms  sunpos+io=1.6ms  TOTAL=221.9ms
--------------------------------------------------

Processing: polarized_003.raw

Final angle (single-pass, n=50 samples): 78.5034°
Yaw candidates (180° ambiguity):
Psi_mes_1 = 290.250°
Psi_mes_2 = 110.250°
  [timing]  read=0.5ms  aolp=114.0ms  cc+svm=72.5ms  sunpos+io=4.2ms  TOTAL=191.1ms
--------------------------------------------------

Processing: polarized_004.raw

Final angle (single-pass, n=50 samples): 78.6132°
Yaw candidates (180° ambiguity):
Psi_m